# PCN Detection — Evaluation Harness

Measures detection accuracy at two levels:
- **Span-level:** did we mask the exact right characters for each annotated PCN?
- **Field-level:** did we correctly identify which fields contain PCNs?

Produces precision, recall, F1, confusion matrix, stratified breakdowns by card type/separator/test category, and an error taxonomy.

---
## Setup

In [ ]:
import json
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pcn_utils import mask_pcns_in_string, PCN_PATTERN

EVAL_DATABASE = "capstone_project.pii_data"
EVAL_TABLE = "pcn_evaluation_corpus"
METRICS_TABLE = "pcn_evaluation_metrics"

TEXT_COLUMNS = ["text_field", "card_number_field", "other_number_field", "metadata_field"]

---
## Load Corpus & Apply Masking

In [ ]:
corpus_df = spark.table(f"{EVAL_DATABASE}.{EVAL_TABLE}").toPandas()
print(f"Loaded {len(corpus_df)} rows from evaluation corpus")
print(f"Categories: {corpus_df['test_category'].value_counts().to_dict()}")

# Apply masking to each text column
for col in TEXT_COLUMNS:
    corpus_df[f"{col}_masked"] = corpus_df[col].apply(
        lambda x: mask_pcns_in_string(x) if pd.notna(x) else x
    )

print("Masking applied to all text columns.")

---
## Evaluation Functions

**Span-level:** for each annotated span, check if every character in `[start, end)` is `X` in the masked output.

**Field-level:** for each field in each row, classify as TP/FP/FN/TN based on whether annotations exist and whether any masking occurred.

**False positives:** scan for positions where `original[i] != masked[i]` that aren't covered by any annotation.

In [ ]:
# Scoring logic lives in pcn_eval.py so method_comparison_benchmark.ipynb uses the same rules
from pcn_eval import evaluate_row, compute_prf

print("Evaluation functions loaded from pcn_eval.")

---
## Run Evaluation

In [ ]:
all_span_results = []
all_field_results = []
all_fp_details = []

for _, row in corpus_df.iterrows():
    sr, fr, fp = evaluate_row(row)
    all_span_results.extend(sr)
    all_field_results.extend(fr)
    all_fp_details.extend(fp)

span_df = pd.DataFrame(all_span_results)
field_df = pd.DataFrame(all_field_results)
fp_df = pd.DataFrame(all_fp_details) if all_fp_details else pd.DataFrame()

print(f"Span evaluations: {len(span_df)}")
print(f"Field evaluations: {len(field_df)}")
print(f"False positive ranges: {len(fp_df)}")

---
## Aggregate Metrics

In [ ]:
# --- Span-level metrics ---
span_tp = len(span_df[span_df["verdict"] == "TP"])
span_fp = len(span_df[span_df["verdict"] == "FP"])
span_fn = len(span_df[span_df["verdict"].isin(["FN", "PARTIAL_FN"])])
span_partial = len(span_df[span_df["verdict"] == "PARTIAL_FN"])
span_p, span_r, span_f1 = compute_prf(span_tp, span_fp, span_fn)

print("=" * 60)
print("SPAN-LEVEL METRICS")
print("=" * 60)
print(f"  True Positives:  {span_tp}")
print(f"  False Positives: {span_fp}")
print(f"  False Negatives: {span_fn} (of which {span_partial} partial)")
print(f"  Precision:       {span_p:.4f}")
print(f"  Recall:          {span_r:.4f}")
print(f"  F1 Score:        {span_f1:.4f}")

# --- Field-level metrics ---
field_tp = len(field_df[field_df["verdict"] == "TP"])
field_fp = len(field_df[field_df["verdict"] == "FP"])
field_fn = len(field_df[field_df["verdict"] == "FN"])
field_tn = len(field_df[field_df["verdict"] == "TN"])
field_p, field_r, field_f1 = compute_prf(field_tp, field_fp, field_fn)

print()
print("=" * 60)
print("FIELD-LEVEL METRICS")
print("=" * 60)
print(f"  True Positives:  {field_tp}")
print(f"  False Positives: {field_fp}")
print(f"  False Negatives: {field_fn}")
print(f"  True Negatives:  {field_tn}")
print(f"  Precision:       {field_p:.4f}")
print(f"  Recall:          {field_r:.4f}")
print(f"  F1 Score:        {field_f1:.4f}")

# --- Confusion matrix ---
print()
print("Field-Level Confusion Matrix:")
print(f"                  Predicted +    Predicted -")
print(f"  Actual +        {field_tp:>10}     {field_fn:>10}")
print(f"  Actual -        {field_fp:>10}     {field_tn:>10}")

---
## Stratified Breakdowns

In [ ]:
# --- By card type (span-level, TP/FN only) ---
tp_fn_spans = span_df[span_df["verdict"].isin(["TP", "FN", "PARTIAL_FN"])].copy()
if len(tp_fn_spans) > 0 and "card_type" in tp_fn_spans.columns:
    print("SPAN-LEVEL BY CARD TYPE")
    print("-" * 60)
    for ct, group in tp_fn_spans.groupby("card_type"):
        if not ct:
            continue
        tp = len(group[group["verdict"] == "TP"])
        fn = len(group[group["verdict"].isin(["FN", "PARTIAL_FN"])])
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        print(f"  {ct:15s}  TP={tp:4d}  FN={fn:4d}  Recall={recall:.4f}")

# --- By separator style ---
print()
print("SPAN-LEVEL BY SEPARATOR")
print("-" * 60)
for sep, group in tp_fn_spans.groupby("separator"):
    if not sep:
        continue
    tp = len(group[group["verdict"] == "TP"])
    fn = len(group[group["verdict"].isin(["FN", "PARTIAL_FN"])])
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"  {sep:15s}  TP={tp:4d}  FN={fn:4d}  Recall={recall:.4f}")

# --- By test category ---
print()
print("FIELD-LEVEL BY TEST CATEGORY")
print("-" * 60)
for cat, group in field_df.groupby("test_category"):
    tp = len(group[group["verdict"] == "TP"])
    fp = len(group[group["verdict"] == "FP"])
    fn = len(group[group["verdict"] == "FN"])
    tn = len(group[group["verdict"] == "TN"])
    p, r, f1 = compute_prf(tp, fp, fn)
    print(f"  {cat:5s}  TP={tp:4d}  FP={fp:4d}  FN={fn:4d}  TN={tn:4d}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")

# --- By test label (detailed) ---
print()
print("FIELD-LEVEL BY TEST LABEL")
print("-" * 60)
for label, group in field_df.groupby("test_label"):
    tp = len(group[group["verdict"] == "TP"])
    fp = len(group[group["verdict"] == "FP"])
    fn = len(group[group["verdict"] == "FN"])
    tn = len(group[group["verdict"] == "TN"])
    total = tp + fp + fn + tn
    print(f"  {label:30s}  TP={tp:3d}  FP={fp:3d}  FN={fn:3d}  TN={tn:3d}  (n={total})")

---
## Error Analysis

In [ ]:
def classify_error(row):
    """Classify a span-level error into an error taxonomy category."""
    if row["verdict"] == "TP":
        return "correct"
    if row["verdict"] == "FP":
        return "luhn_coincidence"
    # FN or PARTIAL_FN
    digits = row.get("digits", "")
    if digits and len(digits) >= 13 and len(digits) <= 19:
        from pcn_utils import passes_luhn as _luhn
        if not _luhn(digits):
            return "luhn_reject"
    if row["verdict"] == "PARTIAL_FN":
        return "partial_mask"
    return "regex_miss"


if len(span_df) > 0:
    span_df["error_type"] = span_df.apply(classify_error, axis=1)

    print("ERROR TAXONOMY")
    print("=" * 60)
    print(span_df["error_type"].value_counts().to_string())

    # Show details of errors
    error_rows = span_df[span_df["error_type"] != "correct"]
    if len(error_rows) > 0:
        print(f"\n{len(error_rows)} error(s) found:")
        display(error_rows.head(20))
    else:
        print("\nNo errors — perfect detection!")

    if len(fp_df) > 0:
        print("\nFalse positive details:")
        display(fp_df.head(20))

---
## Write Metrics to Delta

In [ ]:
from datetime import datetime

metrics_row = {
    "run_timestamp": datetime.now().isoformat(),
    "corpus_size": len(corpus_df),
    "span_tp": int(span_tp),
    "span_fp": int(span_fp),
    "span_fn": int(span_fn),
    "span_precision": round(span_p, 4),
    "span_recall": round(span_r, 4),
    "span_f1": round(span_f1, 4),
    "field_tp": int(field_tp),
    "field_fp": int(field_fp),
    "field_fn": int(field_fn),
    "field_tn": int(field_tn),
    "field_precision": round(field_p, 4),
    "field_recall": round(field_r, 4),
    "field_f1": round(field_f1, 4),
}

metrics_qualified = f"{EVAL_DATABASE}.{METRICS_TABLE}"
(
    spark.createDataFrame([metrics_row])
    .write.format("delta")
    .mode("append")
    .saveAsTable(metrics_qualified)
)

print(f"✔ Metrics appended to '{metrics_qualified}'.")
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"  Span-level:  P={span_p:.4f}  R={span_r:.4f}  F1={span_f1:.4f}")
print(f"  Field-level: P={field_p:.4f}  R={field_r:.4f}  F1={field_f1:.4f}")
print(f"\nPrevious runs:")
display(spark.table(metrics_qualified).orderBy(F.col("run_timestamp").desc()))